In [0]:
import os, sys
from datetime import date, timedelta
sys.path.append(os.path.abspath('..'))

from src.config import SERIES_BCB
from src.extrator import coletar_json, finalizar
from src.http_client import criar_sessao

dbutils.widgets.dropdown('modo', 'diario', ['diario', 'backfill'])
modo = dbutils.widgets.get('modo')

URL = 'https://api.bcb.gov.br/dados/serie/bcdata.sgs.{codigo}/dados'
sessao = criar_sessao()
hoje = date.today()
logs = []

if modo == 'diario':
    janelas = [(hoje - timedelta(days=90), hoje)]
else:
    janelas, inicio = [], date(2016, 1, 1)
    while inicio <= hoje:
        fim = min(date(inicio.year + 4, 12, 31), hoje)  # blocos de até 5 anos
        janelas.append((inicio, fim))
        inicio = fim + timedelta(days=1)

for codigo in SERIES_BCB.values():
    for ini, fim in janelas:
        logs.append(coletar_json(sessao, 'bcb', 'series', URL.format(codigo=codigo), params={
            'formato': 'json',
            'dataInicial': ini.strftime('%d/%m/%Y'),
            'dataFinal': fim.strftime('%d/%m/%Y'),
        }))

finalizar(logs)